![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 36 -- Lab 1: News in Four Sections

A news website publishes stories all day, and an editor puts each one into one of four sections: business, entertainment, sport or tech. Your goal is to build a model that reads a story and picks its section, for the stories nobody has sorted yet. Your file is scored with macro F1.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** English news stories; the first line of every story is its headline. The setup cell leaves three files in `DATA_DIR`: `train.csv` (an `id`, the `text` of the story and its `category`), `test.csv` (the same without the category) and `sample_submission.csv` (the format of the file you hand in).

Your work is checked as you go, and the clock started at the opening time, so start at once and run every checkpoint cell.

---
# Setup

Run the four cells below in order. The first one installs the checkpoint library where it is missing. The second one opens your session with the scorer: replace `PASTE-YOUR-ID` by your own student id before you run it. From that cell on, this notebook records cell runs, cell source and errors for assessment. The third one imports the libraries and the fourth one gets the data.

Your id is yours alone. Do not share a copy of this notebook that holds it.

In [ ]:
# Installs the checkpoint library where it is missing. On Colab that takes a few seconds.
import importlib.util

if importlib.util.find_spec("olympiad") is None:
    !pip -q install olympiad-lab
    importlib.invalidate_caches()


In [ ]:
import os
from olympiad import ck

STUDENT_ID = os.environ.get("OLYMPIAD_STUDENT_ID", "PASTE-YOUR-ID")
ck.start(lab="nlp-01", student_id=STUDENT_ID)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split


In [ ]:
import os

DATA_DIR = os.environ.get("OLYMPIAD_DATA_DIR")      # set when the instructor runs the notebook again
if not DATA_DIR:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("sattamjaltwaim/news-in-four-sections")
print("Data folder:", DATA_DIR)
print(sorted(os.listdir(DATA_DIR))[:20])


---
# The starter solution

Every olympiad task comes with a starter solution: a short notebook that reads the files and hands in a valid file. Its score is the **baseline**, and your points count up from it. This is the starter of this task. Read it before you begin; you may reuse any of its lines.

```python
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

vectorizer = CountVectorizer(max_features=50)
X_train = vectorizer.fit_transform(train["text"])
X_test = vectorizer.transform(test["text"])
model = LogisticRegression(max_iter=1000).fit(X_train, train["category"])

submission = pd.DataFrame({"id": test["id"], "category": model.predict(X_test)})
submission.to_csv("submission.csv", index=False)
print(submission.shape)
```

---
# Part 1 -- Load and look

Every task begins the same way: get the files into memory, then read some stories before you build anything.

## Task 1: Load the three files

Read `train.csv`, `test.csv` and `sample_submission.csv` from `DATA_DIR` with `pd.read_csv` into `train`, `test` and `sample`. Print the three shapes.

Syntax hint (the shape of the call, not the answer):

```python
table = pd.read_csv(os.path.join(DATA_DIR, "name.csv"))
```

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.load(train=train, test=test, sample=sample)


## Task 2: Look at the stories

Store the first five rows of `train` in `first_rows` and the number of rows of `train` in `n_rows`. Then print how many stories each section has, how many words the stories have (the smallest, the mean and the largest count), and the headline of one story of every section. The headline is the first line of the text.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.look(first_rows, size=n_rows)


---
# Part 2 -- Read the task

Before any model comes the task reading: what the metric is, what two answers without a model would score, what the metric rewards, which format the file must have and what you noticed about the task. It is sent once, by the cell after Task 7, and it must arrive before minute 25. It is scored after the lab, so no tick appears.

## Task 3: The metric as a function

The file you hand in is scored with macro F1: the F1 score of every class, then the plain mean of those numbers. Write `macro_f1(y_true, y_pred)` for two lists of labels without calling a library. A class counts when it appears in either list. Print its value for `[0, 1, 2, 2]` and `[0, 2, 2, 2]`; by hand that is (1 + 0 + 0.8) / 3 = 0.6.

Syntax hint (the shape of the call, not the answer):

```python
matches = sum(1 for a, b in zip(list_a, list_b) if a == b)
```

In [ ]:
# Your code here


## Task 4: Two answers without a model

What would two answers that need no model score? `all_class_business` says business for every story. `uniform_random` picks one of the four sections at random for every story. Estimate both from the sections of `train` with your function and store them in the dictionary `trivial` under these two names. For the random answer take the mean of 200 random draws.

Syntax hint (the shape of the call, not the answer):

```python
rng = np.random.default_rng(0)
guesses = rng.choice(options, size).tolist()
```

In [ ]:
# Your code here


## Task 5: What the metric rewards

A model calls every story business, the largest section. What happens to its macro F1?

- **a** It is about 0.3, the share of business stories.
- **b** It is low: three of the four sections get an F1 of 0, and every section counts a quarter.
- **c** It stays high, because business is the largest section.

Store the letter in `rewards`.

In [ ]:
# Your code here


## Task 6: The format of the file

Store the format rules in the dictionary `format_rules`: `n_rows`, the number of rows your file must have, and `id_col`, the name of its id column. Read both from the tables you loaded; do not type the number.

In [ ]:
# Your code here


## Task 7: What you noticed

Write in `facts`, in one to three sentences of your own, what you noticed about this task that its statement does not say and that could change your score.

In [ ]:
# Your code here


Run the cell to send your task reading. It answers with one line and shows no score.

In [ ]:
ck.reading(metric_fn=macro_f1, trivial=trivial, rewards=rewards, format=format_rules, facts=facts)


---
# Part 3 -- A first file

The simplest pipeline that ends in a file comes first: a validation set, features, a model, a score, a file. Aim to hand in your first file within the first hour.

## Task 8: Hold out a validation set

Split `train` into `tr` (80%) and `va` (20%) with `train_test_split`, `random_state=0`, stratified on the section. No story may be in both. Print the two shapes.

Syntax hint (the shape of the call, not the answer):

```python
part_a, part_b = train_test_split(table, test_size=0.3, random_state=0, stratify=table[column])
```

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.split(train_ids=tr["id"], val_ids=va["id"])


## Task 9: Turn the stories into numbers

A model cannot read text: every story must become one row of numbers. Choose how, and build the feature tables `X_tr`, `X_va` and `X_te` from the `text` of `tr`, `va` and `test`. Whatever you fit, fit it on `tr` only. Store the sections in the arrays `y_tr` and `y_va`. Print the shape of `X_tr` and the data types of `X_tr` and `y_tr`.

Syntax hint (the shape of the call, not the answer):

```python
table = vectorizer.fit_transform(texts)      # learns from these texts, then turns them into rows
other = vectorizer.transform(other_texts)    # uses what it learned; learns nothing new
```

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.batch(X_tr[:32], y_tr[:32])


## Task 10: Fit the model

Fit a `LogisticRegression` with `max_iter=1000` on `X_tr` and `y_tr` and store it in `model`. Print `model.classes_`.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.model(model, X_va[:32])


## Task 11: Score the validation set

Predict the section of every story of `X_va` into `val_pred` and store the macro F1 of those predictions in `val_f1`, a number between 0 and 1.

Syntax hint (the shape of the call, not the answer):

```python
score = f1_score(true_labels, predicted_labels, average="macro")
```

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.eval(val_f1, metric="macro_f1")


## Task 12: Write the submission file

Predict `X_te` and write `submission.csv` with the columns of `sample`: the `id` of every test story and your `category`. Leave the index out. Print the shape and the first rows.

Syntax hint (the shape of the call, not the answer):

```python
table.to_csv("name.csv", index=False)
```

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv")


---
# Part 4 -- Look inside your model

A logistic regression on word features is easy to read: it gives every column of your table one weight per section. See what it learned, and where it goes wrong.

## Task 13: The words your model trusts

For every section in `model.classes_`, print the ten words with the largest weight. Row `i` of `model.coef_` holds the weights of section `i`, one per column of your table, and `vectorizer.get_feature_names_out()` gives the word of every column. Do the words make sense?

Syntax hint (the shape of the call, not the answer):

```python
top = np.argsort(values)[::-1][:10]      # positions of the 10 largest values
```

In [ ]:
# Your code here


## Task 14: Read its mistakes

Print every story of `va` that your model got wrong: its true section, the section your model chose and its headline. Then print how many mistakes there are. Read them: would you have got them right?

In [ ]:
# Your code here


---
# Part 5 -- Improve

You have a file on the board. Now make it better, one change at a time, and let the validation set decide which change stays.

## Task 15: Compare settings on the validation set

Try at least three different settings of your features or your model. Fit each on `tr`, score it on `va` with macro F1, and print the scores side by side.

In [ ]:
# Your code here


## Task 16: Hand in your best file

Build your best pipeline, keep its fitted parts in `final_vectorizer` and `final_model`, and write `submission.csv` again. Your last accepted file is your answer, so hand in a new file only when you expect it to be better.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv")


---
# Part 6 -- Challenge: fool your model

Your model has never read your writing. Time to test it.

## Task 17: Write headlines of your own

Make a list `my_headlines` with at least one headline of your own for every section, and print the section `final_model` gives each one. Then add a headline written to fool it: a story of one section that your model puts into another. Who in the room fools it with the shortest headline?

In [ ]:
# Your code here


---
## Task 18: Reflect

Which section was hardest for your model, and what did its mistakes have in common? Which change raised your validation score most, and why do you think it helped? Which headline fooled your model, and what does that tell you about a model that only looks at which words appear?

*Your answers here*

---
# Finish

Run the last cell when you are done. It sends what is still waiting and hands in this notebook.

In [ ]:
ck.finish()

---
## Conclusion

- **Read the task**: macro F1 written by hand, two answers without a model, and the format of the file, all before any model.
- **Text to numbers**: every story became one row of word weights, fitted on the training part only.
- **A first file early**: a validation set, a logistic regression, a validation score and a file within the first hour.
- **Look inside**: the words each section trusts, and the stories the model got wrong.
- **Improve with the validation set**: settings compared on `va`, then the best pipeline handed in.

---

Made By **Sattam Altwaim**